<a href="https://colab.research.google.com/github/Licuao/TelecomX/blob/main/TelecomX_01_API.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import pandas as pd
import requests

In [6]:
url = "https://raw.githubusercontent.com/ingridcristh/challenge2-data-science-LATAM/refs/heads/main/TelecomX_Data.json"
response = requests.get(url)
response.status_code

200

In [7]:
data = response.json()
type(data)

list

In [8]:
df = pd.json_normalize(data)

In [9]:
df.shape[0]

7267

In [10]:
df.dtypes

,0
customerID,object
Churn,object
customer.gender,object
customer.SeniorCitizen,int64
customer.Partner,object
customer.Dependents,object
customer.tenure,int64
phone.PhoneService,object
phone.MultipleLines,object
internet.InternetService,object


In [11]:
df.isna().sum().sort_values(ascending=False)

,0
customerID,0
Churn,0
customer.gender,0
customer.SeniorCitizen,0
customer.Partner,0
customer.Dependents,0
customer.tenure,0
phone.PhoneService,0
phone.MultipleLines,0
internet.InternetService,0


In [12]:
df["account.Charges.Total"].unique()[:10]

array(['593.3', '542.4', '280.85', '1237.85', '267.4', '571.45',
       '7904.25', '5377.8', '340.35', '5957.9'], dtype=object)

In [13]:
df["account.Charges.Total"] = (
    df["account.Charges.Total"]
    .replace(" ", pd.NA)
    .astype(float)
)

TypeError: float() argument must be a string or a real number, not 'NAType'

In [14]:
df["account.Charges.Total"] = pd.to_numeric(
    df["account.Charges.Total"],
    errors="coerce"
)

In [15]:
df["account.Charges.Total"].dtype

dtype('float64')

In [16]:
df["account.Charges.Total"].isna().sum()


np.int64(11)

In [17]:
df["account.Charges.Monthly"] = pd.to_numeric(
    df["account.Charges.Monthly"],
    errors="coerce"
)

In [18]:
df["account.Charges.Monthly"].dtype

dtype('float64')

In [19]:
df["customer.tenure"].dtype

dtype('int64')

In [20]:
df["customer.tenure"] = pd.to_numeric(
    df["customer.tenure"],
    errors="coerce"
)

In [21]:
df["customer.SeniorCitizen"].value_counts()

,count
customer.SeniorCitizen,
0,6085
1,1182


In [22]:
df.select_dtypes(include=["int64", "float64"]).columns

Index(['customer.SeniorCitizen', 'customer.tenure', 'account.Charges.Monthly',
       'account.Charges.Total'],
      dtype='object')

In [23]:
df.select_dtypes(include=["int64", "float64"]).describe().T

,count,mean,std,min,25%,50%,75%,max
customer.SeniorCitizen,7267.0,0.162653,0.369074,0.00,0.000,0.0,0.000,1.00
customer.tenure,7267.0,32.346498,24.571773,0.00,9.000,29.0,55.000,72.00
account.Charges.Monthly,7267.0,64.720098,30.129572,18.25,35.425,70.3,89.875,118.75
account.Charges.Total,7256.0,2280.634213,2268.632997,18.80,400.225,1391.0,3785.300,8684.80


In [24]:
df["customer.tenure"].describe()

,customer.tenure
count,7267.000000
mean,32.346498
std,24.571773
min,0.000000
25%,9.000000
50%,29.000000
75%,55.000000
max,72.000000


In [25]:
df["account.Charges.Monthly"].describe()

,account.Charges.Monthly
count,7267.000000
mean,64.720098
std,30.129572
min,18.250000
25%,35.425000
50%,70.300000
75%,89.875000
max,118.750000


In [26]:
df["account.Charges.Total"].describe()

,account.Charges.Total
count,7256.000000
mean,2280.634213
std,2268.632997
min,18.800000
25%,400.225000
50%,1391.000000
75%,3785.300000
max,8684.800000


In [27]:
df["Churn"].value_counts(normalize=True) * 100

,proportion
Churn,
No,71.198569
Yes,25.719004
,3.082427


In [28]:
df["internet.InternetService"].value_counts(normalize=True) * 100

,proportion
internet.InternetService,
Fiber optic,44.007156
DSL,34.236962
No,21.755883


Final

In [29]:
# Distribución de churn
churn_dist = df["Churn"].value_counts(normalize=True) * 100
churn_dist

,proportion
Churn,
No,71.198569
Yes,25.719004
,3.082427


In [31]:
def churn_por_categoria(df, columna):
    return (
        df.groupby(columna)["Churn"]
        .value_counts(normalize=True)
        .unstack()
        .fillna(0) * 100
    )

In [32]:
churn_por_categoria(df, "account.Contract")

Churn,,No,Yes
account.Contract,,,
Month-to-month,3.245943,55.430712,41.323346
One year,3.028308,86.043450,10.928242
Two year,2.753873,94.492255,2.753873


In [33]:
churn_por_categoria(df, "internet.InternetService")

Churn,,No,Yes
internet.InternetService,,,
DSL,2.692926,78.858521,18.448553
Fiber optic,3.189493,56.253909,40.556598
No,3.478811,89.373814,7.147375


In [34]:
churn_por_categoria(df, "internet.TechSupport")

Churn,,No,Yes
internet.TechSupport,,,
No,3.042993,56.588498,40.368509
No internet service,3.478811,89.373814,7.147375
Yes,2.851711,82.414449,14.733840


In [35]:
df.groupby("Churn")[[
    "customer.tenure",
    "account.Charges.Monthly",
    "account.Charges.Total"
]].describe().T

Churn                                                No          Yes
customer.tenure         count   224.000000  5174.000000  1869.000000
                        mean     31.571429    37.569965    17.979133
                        std      24.998552    24.113777    19.531123
                        min       1.000000     0.000000     1.000000
                        25%       7.000000    15.000000     2.000000
                        50%      29.000000    38.000000    10.000000
                        75%      56.000000    61.000000    29.000000
                        max      72.000000    72.000000    72.000000
account.Charges.Monthly count   224.000000  5174.000000  1869.000000
                        mean     63.412277    61.265124    74.441332
                        std      31.388712    31.092648    24.666053
                        min      18.750000    18.250000    18.850000
                        25%      28.425000    25.100000    56.150000
                        50%      69.100000    64.425000    79.650000
                        75%      90.412500    88.400000    94.200000
                        max     115.550000   118.750000   118.350000
account.Charges.Total   count   224.000000  5163.000000  1869.000000
                        mean   2196.933705  2555.344141  1531.796094
                        std    2329.961954  2329.456984  1890.822994
                        min      18.900000    18.800000    18.850000
                        25%     351.037500   577.825000   134.500000
                        50%    1163.175000  1683.600000   703.550000
                        75%    3562.862500  4264.125000  2331.300000
                        max    8425.300000  8672.450000  8684.800000